# 기능개발 문제 풀이 정리

이 노트북은 아래 파이썬 코드를 기준으로,

- 왜 이렇게 푸는지
- `current` 와 `cnt` 가 무슨 역할인지
- 예시에서 어떻게 동작하는지
- 같은 로직을 Java로 어떻게 옮기는지

를 한 단계씩 설명합니다.


## 1. 파이썬 코드

```python
import math

def solution(progresses, speeds):
    answer = []
    days = []
    for p, s in zip(progresses, speeds):
        days.append(math.ceil((100 - p) / s))

    current = days[0]
    cnt = 1

    for i in range(1, len(days)):
        if days[i] <= current:
            cnt += 1
        else:
            answer.append(cnt)
            current = days[i]
            cnt = 1

    answer.append(cnt)
    return answer
```


## 2. 핵심 아이디어

이 문제는 **각 기능이 완성되는 날짜를 먼저 구한 뒤**, 앞 기능을 기준으로 뒤 기능들이 **같이 배포 가능한지 묶는 문제**입니다.

중요한 점은 다음입니다.

1. 기능의 **순서는 바뀌지 않습니다**.
2. 뒤 기능이 먼저 끝나도, 앞 기능이 안 끝났으면 기다려야 합니다.
3. 그래서 정렬이나 swap이 아니라, **앞 기능의 완료일을 기준으로 그룹을 세는 방식**으로 풀어야 합니다.


## 3. `days` 배열 만들기

각 기능이 배포 가능해지기까지 며칠 걸리는지 계산합니다.

공식은 아래와 같습니다.

- 남은 작업량 = `100 - progress`
- 하루 작업량 = `speed`
- 필요한 날짜 = `ceil((100 - progress) / speed)`

`ceil` 을 쓰는 이유는 **소수점이 생기면 하루를 더 써야 하기 때문**입니다.


In [ ]:
import math

progresses = [93, 30, 55]
speeds = [1, 30, 5]

days = []
for p, s in zip(progresses, speeds):
    days.append(math.ceil((100 - p) / s))

days


위 결과는 `[7, 3, 9]` 입니다.

즉,

- 첫 번째 기능: 7일
- 두 번째 기능: 3일
- 세 번째 기능: 9일

이 뜻입니다.


## 4. `current` 와 `cnt` 의 의미

- `current`: 현재 배포 묶음의 기준 완료일
- `cnt`: 현재 배포 묶음에 들어 있는 기능 수

처음에는 첫 번째 기능이 기준이 되므로,

- `current = days[0]`
- `cnt = 1`

로 시작합니다.


## 5. 왜 `days[i] <= current` 이면 같이 배포할까?

예를 들어 현재 기준 완료일이 `7`일이라고 합시다.

- 다음 기능이 `3`일이면: 이미 더 빨리 끝나 있으므로 같이 배포 가능
- 다음 기능이 `7`일이면: 같은 날 끝나므로 같이 배포 가능
- 다음 기능이 `9`일이면: 기준보다 늦게 끝나므로 다음 배포로 넘겨야 함

그래서 조건은 아래처럼 됩니다.

- `days[i] <= current` → 현재 묶음에 포함
- `days[i] > current` → 새 묶음 시작


In [ ]:
import math

def solution(progresses, speeds):
    answer = []
    days = []
    for p, s in zip(progresses, speeds):
        days.append(math.ceil((100 - p) / s))

    print('완료까지 걸리는 날짜:', days)

    current = days[0]
    cnt = 1
    print(f'초기 기준 current={current}, cnt={cnt}')

    for i in range(1, len(days)):
        print(f'\n현재 확인 중인 기능의 완료일: days[{i}] = {days[i]}')

        if days[i] <= current:
            cnt += 1
            print(f'현재 기준일 {current} 이하이므로 같이 배포 가능 -> cnt={cnt}')
        else:
            answer.append(cnt)
            print(f'현재 기준일 {current}보다 늦게 끝남 -> 지금까지 {cnt}개 배포')
            current = days[i]
            cnt = 1
            print(f'새 기준 current={current}, cnt={cnt}')

    answer.append(cnt)
    print(f'마지막 묶음 {cnt}개 추가')
    return answer

solution([93, 30, 55], [1, 30, 5])


## 6. 한 줄씩 해석

```python
answer = []
```
최종 배포 개수를 저장할 리스트입니다.

```python
days = []
```
각 기능이 완료되기까지 걸리는 날짜를 저장합니다.

```python
for p, s in zip(progresses, speeds):
    days.append(math.ceil((100 - p) / s))
```
각 기능마다 남은 날짜를 계산해서 `days`에 넣습니다.

```python
current = days[0]
cnt = 1
```
첫 기능을 기준으로 첫 배포 묶음을 시작합니다.

```python
for i in range(1, len(days)):
```
두 번째 기능부터 끝까지 확인합니다.

```python
if days[i] <= current:
    cnt += 1
```
현재 기준일보다 빨리 끝나거나 같은 날 끝나면 같이 배포합니다.

```python
else:
    answer.append(cnt)
    current = days[i]
    cnt = 1
```
현재 기준일보다 늦게 끝나면 기존 묶음을 종료하고, 새 묶음을 시작합니다.

```python
answer.append(cnt)
```
반복문이 끝난 뒤 마지막 묶음을 넣어줍니다.


## 7. Java 코드로 바꾸기

파이썬의 리스트는 Java에서는 `ArrayList<Integer>` 로 바꿔 생각하면 편합니다.

또한 `math.ceil` 은 Java에서 `Math.ceil` 을 사용합니다.


```java
import java.util.*;

class Solution {
    public int[] solution(int[] progresses, int[] speeds) {
        List<Integer> answer = new ArrayList<>();
        List<Integer> days = new ArrayList<>();

        for (int i = 0; i < progresses.length; i++) {
            int day = (int)Math.ceil((100.0 - progresses[i]) / speeds[i]);
            days.add(day);
        }

        int current = days.get(0);
        int cnt = 1;

        for (int i = 1; i < days.size(); i++) {
            if (days.get(i) <= current) {
                cnt++;
            } else {
                answer.add(cnt);
                current = days.get(i);
                cnt = 1;
            }
        }

        answer.add(cnt);

        int[] result = new int[answer.size()];
        for (int i = 0; i < answer.size(); i++) {
            result[i] = answer.get(i);
        }

        return result;
    }
}
```


## 8. Java에서 특히 주의할 점

### 1) `100.0` 으로 써야 하는 이유

```java
(100.0 - progresses[i]) / speeds[i]
```

이렇게 `100.0` 으로 써야 **실수 나눗셈**이 됩니다.

만약 `100` 으로만 쓰면 정수 나눗셈이 되어 소수점이 잘릴 수 있습니다.

### 2) `ArrayList<Integer>` 를 쓰는 이유

파이썬은 리스트 크기가 자동으로 늘어나지만, Java의 배열은 크기가 고정입니다.
그래서 중간 결과를 담을 때는 `ArrayList<Integer>` 가 편합니다.

### 3) 마지막에 `int[]` 로 바꾸는 이유

프로그래머스 Java 정답 형식은 보통 `int[]` 이기 때문에,
마지막에 `List<Integer>` 를 `int[]` 로 변환해 줍니다.


## 9. 최종 정리

이 문제의 핵심은 아래 한 문장으로 정리할 수 있습니다.

**기능 완료일을 구한 뒤, 앞 기능의 완료일을 기준으로 뒤 기능들을 묶어서 배포 개수를 세는 문제**

즉,

- 정렬하는 문제 아님
- swap 하는 문제 아님
- 순서를 유지한 채 그룹을 세는 문제

입니다.
